In [ ]:
# Cell 1: Imports and DB connection
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

# Add project root to sys.path so ib_trader is importable when running from notebooks/
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

from ib_trader.tracker import init_tracker_db, load_backtest_benchmarks, register_strategy
from ib_trader.report import (
    snapshot_report,
    get_trades,
    get_positions,
    get_performance,
    get_tax_summary,
    compare_vs_backtest,
    get_ic_series,
)

conn = init_tracker_db()
STRATEGY = 'fundamentals_alpha'

# Register strategy if not already done (only needed once)
# register_strategy(conn, STRATEGY, description='XGBoost fundamentals alpha', benchmark='SPY')

# Register backtest benchmark stats (only needed once)
# load_backtest_benchmarks(conn, STRATEGY, 'vw_gr_top_n_25', {
#     'cagr': 0.252, 'ann_vol': 0.225, 'sharpe': 1.35, 'sortino': 1.62,
#     'max_dd': -0.216, 'beta': 1.32, 'alpha': 0.10, 'win_rate': 0.67,
# })

print(f'Connected to tracker DB: {os.getenv("IB_TRACKER_DB", "data/ib_tracker.duckdb")}')

In [ ]:
# Cell 2: Full strategy snapshot report
# Pass client=ib_client to fetch live NAV and prices from IB
snapshot_report(conn, STRATEGY)

In [ ]:
# Cell 3: Trade history
# Filter by ticker or date range:
#   get_trades(conn, STRATEGY, ticker='AAPL')
#   get_trades(conn, STRATEGY, from_date=date(2026, 1, 1))
trades = get_trades(conn, STRATEGY)
trades

In [ ]:
# Cell 4: Tax summary (realized P&L with ST/LT breakdown)
# Filter by year:  get_tax_summary(conn, STRATEGY, year=2026)
import pandas as pd
pd.set_option('display.float_format', '${:,.2f}'.format)

tax = get_tax_summary(conn, STRATEGY, year=2026)
if not tax.empty:
    print(f"Total ST realized: ${tax[~tax['is_long_term']]['realized_pnl'].sum():,.2f}")
    print(f"Total LT realized: ${tax[tax['is_long_term']]['realized_pnl'].sum():,.2f}")
    print(f"Estimated tax:     ${tax['tax_owed'].sum():,.2f}")
tax

In [ ]:
# Cell 5: Live vs backtest performance comparison
cmp = compare_vs_backtest(conn, STRATEGY)
if cmp.empty:
    print('No backtest benchmarks registered.')
    print('Call load_backtest_benchmarks(conn, STRATEGY, "vw_gr_top_n_25", {...}) first.')
else:
    cmp

In [ ]:
# Cell 6: Model IC time series (score rank vs forward 30-day return)
# To update forward returns for snapshots older than 30 days:
#   from ib_trader.tracker import update_forward_returns
#   update_forward_returns(conn, STRATEGY)

import matplotlib.pyplot as plt

ic_df = get_ic_series(conn, STRATEGY)
if ic_df.empty:
    print('No IC data yet. Record score snapshots and run update_forward_returns().')
else:
    print(f'Mean IC: {ic_df["ic"].mean():+.4f}  (N={len(ic_df)} months)')
    print(f'% months IC > 0: {(ic_df["ic"] > 0).mean():.1%}')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(ic_df['snapshot_date'], ic_df['ic'],
           color=['steelblue' if v > 0 else 'tomato' for v in ic_df['ic']])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(ic_df['ic'].mean(), color='orange', linestyle='--', linewidth=1.5, label=f"Mean IC")
    ax.set_xlabel('Snapshot date')
    ax.set_ylabel('Spearman IC')
    ax.set_title(f'{STRATEGY} — Model IC (score rank vs 30-day forward return)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    ic_df